# Lab 3 — Data Preprocessing & Exploratory Data Analysis

**Student:** Hitesh Gupta | **Roll No.:** IC-2k23-39

This notebook performs complete preprocessing and EDA on two datasets:
- `placement.csv` — student placement records
- `iris.csv` — classic Iris flower dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
DATA_DIR = '../datasets'

placement = pd.read_csv(f'{DATA_DIR}/placement.csv')
iris = pd.read_csv(f'{DATA_DIR}/iris.csv')

print('Placement shape:', placement.shape)
print(placement.head())
print('\nIris shape:', iris.shape)
print(iris.head())

## 1. Missing Value Analysis

In [ ]:
print('Placement missing values:')
print(placement.isnull().sum())
print('\nIris missing values:')
print(iris.isnull().sum())

# Visualise missingness
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(placement.isnull(), cbar=False, ax=axes[0], cmap='viridis')
axes[0].set_title('Placement — Missing Values')
sns.heatmap(iris.isnull(), cbar=False, ax=axes[1], cmap='viridis')
axes[1].set_title('Iris — Missing Values')
plt.tight_layout()
plt.savefig('outputs/missing_values.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Handling Missing Values & Outliers

In [ ]:
# Numerical columns -> median imputation; categorical -> mode
num_cols = placement.select_dtypes(include=[np.number]).columns
cat_cols = placement.select_dtypes(include=['object']).columns

for col in num_cols:
    placement[col].fillna(placement[col].median(), inplace=True)
for col in cat_cols:
    placement[col].fillna(placement[col].mode()[0], inplace=True)

print('Missing after imputation:', placement.isnull().sum().sum())

# Outlier detection via IQR on numeric features
fig, axes = plt.subplots(1, len(num_cols), figsize=(4 * len(num_cols), 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=placement[col], ax=ax, color='lightblue')
    ax.set_title(col)
plt.tight_layout()
plt.savefig('outputs/outliers.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Encoding Categorical Variables

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Label encoding for ordinal-like columns
le = LabelEncoder()
for col in cat_cols:
    placement[col] = le.fit_transform(placement[col].astype(str))

print('Encoded columns:', list(cat_cols))
print(placement.dtypes)

# One-hot encoding example on a low-cardinality column
placement_oh = pd.get_dummies(placement, columns=[cat_cols[0]], drop_first=True)
print('\nShape after one-hot:', placement_oh.shape)

## 4. Feature Scaling & Selection

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif

scaler = StandardScaler()
X_scaled = scaler.fit_transform(placement[num_cols])

# Feature importance via mutual information
target_col = [c for c in placement.columns if 'status' in c.lower()
              or 'placed' in c.lower() or 'package' in c.lower()]
if target_col:
    y = placement[target_col[0]]
    mi = mutual_info_classif(X_scaled, y)
    mi_df = pd.DataFrame({'Feature': num_cols, 'MI Score': mi})\
               .sort_values('MI Score', ascending=False)
    print(mi_df.to_string(index=False))
    mi_df.plot.bar(x='Feature', y='MI Score', legend=False, figsize=(8, 4))
    plt.title('Feature Importance (Mutual Information)')
    plt.tight_layout()
    plt.savefig('outputs/feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. EDA Visualisations — Iris Dataset

In [ ]:
# Pairplot
sns.pairplot(iris, hue=iris.columns[-1], diag_kind='kde', height=2)
plt.savefig('outputs/iris_pairplot.png', dpi=300, bbox_inches='tight')
plt.show()

# Correlation heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(iris.select_dtypes(include=[np.number]).corr(),
            annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Iris Feature Correlation')
plt.tight_layout()
plt.savefig('outputs/iris_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. EDA Visualisations — Placement Dataset

In [ ]:
# Distribution of key numeric features
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, num_cols[:6]):
    sns.histplot(placement[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(col)
plt.tight_layout()
plt.savefig('outputs/placement_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(placement[num_cols].corr(), annot=True, cmap='coolwarm',
            fmt='.2f', linewidths=0.5)
plt.title('Placement Feature Correlation')
plt.tight_layout()
plt.savefig('outputs/placement_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary

- Missing values were imputed using median (numeric) and mode (categorical).
- Outliers were identified via IQR boxplots.
- Categorical features were label-encoded and one-hot encoded.
- Features were scaled with `StandardScaler`.
- Mutual information ranked feature importance.
- EDA visualisations reveal class separability (Iris) and placement trends.

**Impact of preprocessing:** encoding converts categorical data into a
numerical form ML models require; scaling prevents features with large
ranges from dominating distance-based algorithms; imputation preserves
sample size that would otherwise be lost through row deletion.